<a href="https://colab.research.google.com/github/heitorduenhas/Aurora-Siger/blob/main/Aurora_Siger_Relatorio_de_pr%C3%A9_decolagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1.1 Organização e descrição da telemetria**
### Dados:  
Temperatura do motor:  
Faixa segura: 20°C a 45°C  

Monitora o comportamento térmico da combustão.

* Temperatura alta → risco de falha estrutural ou explosão  
* Temperatura baixa → combustão ineficiente (perda de empuxo)


---


Pressão do motor:  
Faixa segura: 80 bar a 160 bar  

Indica a força interna da combustão.  

* Pressão baixa → empuxo insuficiente  
* Pressão alta → risco de ruptura da câmara

---


Nível de bateria:  
Faixa segura: 70% a 100%

Alimenta todos os sistemas eletrônicos.  

* Controla sensores, comunicação e navegação  
* Falha de energia = perda total de controle


---


Tensão eletrica:  
Faixa segura: 24V a 28V  

Garante que os sistemas recebam energia estável.  

Tensão instável pode causar:  

* Reinício de sistemas  
* Leituras erradas de sensores  

---

Temperatura externa:  
Faixa segura: -10°C a 35°C  

Afeta combustível, materiais e sensores.  

Muito frio → combustível pode não vaporizar corretamente  
Muito quente → sobrecarga térmica

---

Velocidade do vento:  
Faixa segura: 0km/h a 40km/h  

Impacta diretamente a trajetória inicial.  

* Vento forte → desvio de rota  
* Pode gerar instabilidade logo após o lançamento

---

Umidade:  
Faixa segura: 10% a 80%  

Afeta componentes eletrônicos e combustão.

* Umidade alta → risco de curto-circuito  
* Condensação em sensores

---

Pressão atmosférica:  
Faixa segura: 950 hPa a 1050 hPa  

Define condições do ar ao redor.  

* Afeta desempenho do motor  
* Importante para calibração de sensores  

---

Integridade estrutural:  
Valor seguro: OK (1)  

Avalia se o foguete está fisicamente seguro.

* Pequenas falhas podem se tornar catastróficas sob força extrema  
* Durante a decolagem, forças são enormes

---

Vibração:  
Faixa segura: 0 m/s² a 5 m/s²

Detecta instabilidade mecânica.

Vibração excessiva pode:  

* Soltar componentes  
* Danificar sensores

---

Status dos sensores:  
valor seguro: OK (1)  

Garante que os dados são confiáveis.

* Sensor falhando = decisão baseada em dado errado

---

Latência da comunicacao:  
Faixa segura: 0 ms a 100 ms  

Tempo de resposta do sistema.

* Alta latência → atraso em decisões críticas  
* Pode impedir abortamento a tempo

---

Combustível restante:  
Faixa segura: 90% a 100%  

Garante autonomia da missão.

* Combustível insuficiente → missão falha  
* Pode causar queda após decolagem



---


# **1.2 Algoritmo de verificação**
#### Lógica para verificação:

    INICIO

    DEFINIR LIMITES COMO
      temperatura_motor → (20, 45)  
      pressao_motor → (80, 160)  
      nivel_bateria → (70, 100)  
      tensao_eletrica → (24, 28)  
      temperatura_externa → (-10, 35)  
      velocidade_vento → (0, 40)  
      umidade → (10, 80)  
      pressao_atmosferica → (950, 1050)  
      vibracao → (0, 5)  
      latencia_comunicacao → (0, 100)  
      combustivel_restante → (90, 100)  

    DEFINIR VALORES_FIXOS COMO
      integridade_estrutural → 1  
      status_sensor → 1  

    
    FUNCAO verificar_anomalia(linha)
      
      PARA CADA coluna E (min, max) EM LIMITES FACA
        valor = linha[coluna]
        
        SE valor < min OU valor > max ENTAO
          RETORNAR VERDADEIRO
        FIM_SE

      FIM_PARA

      PARA CADA coluna E valor_esperado EM VALORES_FIXOS FACA
        
        SE linha[coluna] ≠ valor_esperado ENTAO
          RETORNAR VERDADEIRO
        FIM_SE
        
      FIM_PARA  

      RETORNAR FALSO
    FIM_FUNCAO

    FUNCAO analisar_decolagem(tabela, numero)

      nome = "Aurora-Siger " + numero

      SE nome NAO ESTA NA tabela ENTAO
        RETORNAR "INVALIDA"
      FIM_SE

      linha = tabela[nome]

      SE verificar_anomalia(linha) FOR VERDADEIRO ENTAO
        RETORNAR "ABORTADA"
      FIM_SE
      SENAO ENTAO
        RETORNAR "PRONTA"
      FIM_SENAO
    FIM_FUNCAO



---


# **1.3 Script em Python**

In [5]:
import pandas as pd

# DEFINIÇÃO DOS VALORES
# Fácil de fazer manutenção dos dados

LIMITES = {
    "temperatura_motor": (20, 45),
    "pressao_motor": (80, 160),
    "nivel_bateria": (70, 100),
    "tensao_eletrica": (24, 28),
    "temperatura_externa": (-10, 35),
    "velocidade_vento": (0, 40),
    "umidade": (10, 80),
    "pressao_atmosferica": (950, 1050),
    "vibracao": (0, 5),
    "latencia_comunicacao": (0, 100),
    "combustivel_restante": (90, 100),
}

VALORES_FIXOS = {
    "integridade_estrutural": 1,
    "status_sensor": 1,
}


# FUNÇÕES DE NEGÓCIO

# Retorna um "dataset" com índice (Aurora-Siger 1, Aurora-Siger 2, ...)
def carregar_dados(caminho_arquivo):
    df = pd.read_csv(caminho_arquivo)
    return df.set_index("num_decolagem")

# verifica se a linha contém alguma anomalia
# Encerra ao achar a primeira anomalia
def verificar_anomalia(row):
    # verifica faixas numéricas
    for item, (min_val, max_val) in LIMITES.items():
        valor = row[item]
        if valor < min_val or valor > max_val:
            return True

    # Verifica valores fixos
    for item, valor_esperado in VALORES_FIXOS.items():
        if row[item] != valor_esperado:
            return True

    return False # caso não tenha anomalia

# Procura a linha que contém o número (Caso não tenha, retorna INDISPONÍVEL)
# Retorna Pronta ou Abortada
def analisar_decolagem(df, numero):
    nome = f"Aurora-Siger {numero}"

    if nome not in df.index:
        return "INDISPONÍVEL"

    row = df.loc[nome]

    if verificar_anomalia(row):
        return "DECOLAGEM ABORTADA"
    else:
        return "PRONTA PARA DECOLAR"


# INTERFACE (INTERAÇÃO COM USUÁRIO)

def executar_sistema():
    df = carregar_dados("dataset_telemetria.csv")

    while True:
        entrada = input("Digite o número da Aurora-Siger (0 para encerrar): ")

        # Trata o erro caso o usuário digite algo diferente de um número
        try:
            numero = int(entrada)
        except ValueError:
            print("Entrada inválida")
            print("//////////////////////////////////\n")
            continue

        # Encerra o programa caso digite 0
        if numero == 0:
            print("ENCERRADO")
            break

        status = analisar_decolagem(df, numero)

        if status == "INDISPONÍVEL":
            print("Aurora-Siger INDISPONÍVEL")
        else:
            print(f"Aurora-Siger {numero} // {status}")

        print("//////////////////////////////////\n")


# PONTO DE ENTRADA

if __name__ == "__main__":
    executar_sistema()

Digite o número da Aurora-Siger (0 para encerrar): 10
Aurora-Siger 10 // PRONTA PARA DECOLAR
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): 16
Aurora-Siger 16 // DECOLAGEM ABORTADA
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): 35
Aurora-Siger 35 // PRONTA PARA DECOLAR
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): 122
Aurora-Siger INDISPONÍVEL
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): abc
Entrada inválida
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): 50
Aurora-Siger 50 // PRONTA PARA DECOLAR
//////////////////////////////////

Digite o número da Aurora-Siger (0 para encerrar): 0
ENCERRADO




---


# **1.4 Análise Energética**

Grandezas:

Capacidade total (KWh)  
Carga atual (%)  
Consumo na decolagem (KWh)  
Perdas energéticas (%)


---


A energia disponível é calculada com base em:  
    
    energia_diponivel = capacidade_total * (carga_atual / 100)  

Pois a a porcentagem de carga atual aplicada sobre a capacidade total da bateria define a quantidade (em KWh) de energia disponível. Exemplo:

    energia_diponivel = 1000 * (80 / 100)
    energia_disponivel = 1000 * 0,8
    energia_disponivel = 800 KWh

Já o consumo real é calculado a partir daqui:

    consumo_real = consumo_estimado * (1 + perdas / 100)

Pois, primeiramente, temos o consumo estimado durante a decolagem. Porém, nenhum sistema é 100% eficiente e tende a ter perdas energéticas. Assim:

    consumo_real = 600 * (1 + 10 / 100)
    consumo_real = 600 * (1 + 0,1)
    consumo_real = 600 * 1,1
    consumo real = 660 KWh

Dessa forma, após calcularmos a energia disponível e o consumo real, precisamos fazer uma operação lógica, onde as comparamos e, caso a energia disponível seja maior que o consumo real, a nave pode decolar, caso contrario a decolagem deve ser abortada:

    SE energia_disponível >= consumo_real ENTAO
      RETORNE "Energia suficiente para decolagem"
    FIM_SE
    SENAO ENTAO
      RETORNE "Energia insuficiente. ABORTAR!!!"
    FIM_SENAO

In [4]:
def calc_energia_disponivel(capaci_total, carga_atual):
  return capaci_total * (carga_atual / 100)

def calc_consumo_real(consumo, perdas):
  perda_real = perdas / 100
  return consumo * (perda_real + 1)

def calc_analise_energetica(energia_disponivel, consumo_real):
  if energia_disponivel >= consumo_real:
    return "Energia suficiente para decolagem"
  else:
    return "Energia insuficiente. ABORTAR!!"

def executar_sistema():
  capaci_total = int(input("Digite a capacidade total(KWh): "))
  carga_atual = int(input("Digite a carga atual(%): "))
  consumo = int(input("Digite o consumo na decolagem(KWh): "))
  perdas = int(input("Digite as perdas energéticas estimadas(%): "))

  energia_disponivel = calc_energia_disponivel(capaci_total, carga_atual)
  consumo_real = calc_consumo_real(consumo, perdas)
  analise_energetica = calc_analise_energetica(energia_disponivel, consumo_real)

  print(f"\nEnergia disponível: {energia_disponivel} KWh")
  print(f"Consumo real: {consumo_real} KWh")
  print(analise_energetica)


if __name__ == "__main__":
    executar_sistema()

Digite a capacidade total(KWh): 1000
Digite a carga atual(%): 80
Digite o consumo na decolagem(KWh): 600
Digite as perdas energéticas estimadas(%): 10

Energia disponível: 800.0 KWh
Consumo real: 660.0 KWh
Energia suficiente para decolagem


# **1.5 Análise assistida por I.A.**

##📊 **1. Classificação dos Dados**

###Podemos dividir o dataset em 3 grupos principais:

###✅ A. Dados Nominais (identificação)  
    num_decolagem → identificação da missão  

###📈 B. Dados Quantitativos (contínuos)  
    Variáveis que medem condições físicas:

    temperatura_motor  
    pressao_motor  
    nivel_bateria  
    tensao_eletrica  
    temperatura_externa  
    velocidade_vento  
    umidade  
    pressao_atmosferica  
    vibracao  
    latencia_comunicacao  
    combustivel_restante  

###⚙️ C. Dados Binários (estado/sistema)  
    integridade_estrutural (0 = falha, 1 = OK)  
    status_sensor (0 = falha, 1 = OK)  


---


##⚠️ 2. Identificação de Anomalias

###🔎 Padrão geral observado:  
Registros 1–12 e 25–50 → comportamento normal  
Registros 13–24 → zona crítica/anômala  

###🚨 Principais anomalias detectadas:

###🔥 **Temperatura do motor muito alta**  
Normal: ~26–36°C  
Anômalo: 50–75°C  
Casos:

Aurora-Siger 13, 15, 19, 24

👉 **Indica superaquecimento**

###💥 **Pressão do motor elevada**  
Normal: ~100–150  
Anômalo: 165–190  
Casos:  
13–24 praticamente todos

👉 **Risco de explosão ou falha estrutural**

###📉 **Latência extremamente alta**  
Normal: ~10–70 ms  
Anômalo: 200–275 ms  
Casos:  
13–24

👉 **Pode causar perda de controle em tempo real**

###🌪️ **Condições ambientais extremas**  
Vento:  
Normal: até ~30  
Anômalo: 50–75  
Umidade:  
Normal: ~30–75  
Anômalo: 85–98  
Temperatura externa:  
Normal: ~5–34°C  
Anômalo: 40–58°C  

👉 **Indica ambiente hostil para lançamento**

### 📳 **Vibração excessiva**  
Normal: 1–4  
Anômalo: 6–9  

👉 **Indica instabilidade estrutural**

###❌ **Falhas críticas de sistema**  
integridade_estrutural = 0  
status_sensor = 0  

Casos:

Aurora-Siger: 13, 15, 17, 19, 21, 24

👉 **Situação CRÍTICA (não pode decolar)**


---


##🚀 3. Sugestões de Riscos  

##🔴 **Riscos Críticos (ALTO IMPACTO)**  

###💣 1. Explosão do motor  

Causas:

Alta pressão + alta temperatura

👉 **Observado nos registros 13–24**

###🧱 2. Falha estrutural

Causas:

Vibração alta
Integridade = 0

👉 **Pode causar desintegração em voo**

###📡 3. Perda de comunicação

Causas:

Latência muito alta

👉 **Consequência: Foguete fica sem controle**  

##🟠 **Riscos Moderados**  

###🌪️ 4. Desvio de trajetória  

Causas:

Vento elevado  

###⚡ 5. Instabilidade elétrica

Causas:

Tensão acima do normal (30–34)  

## 🟡 **Riscos Operacionais**  

###🔋 6. Eficiência comprometida  

Causas:

Combustível baixo (75–85 em alguns casos)  

# 📌Conclusão Geral:  
✔️ A maioria dos dados está dentro da normalidade  
⚠️ Registros 13–24 representam um cenário crítico completo  
🚫 Esses casos devem ser classificados como:  

👉 **"NÃO APTO PARA DECOLAGEM"**



---


# **1.6 Reflexão Crítica**

O projeto desenvolvido evidencia, de forma clara, a importância da ética e da responsabilidade no contexto tecnológico, especialmente quando aplicado à exploração espacial. Ao criar um sistema capaz de analisar dados de telemetria e decidir se uma missão deve ser abortada ou autorizada para decolagem, os desenvolvedores assumem uma posição de grande responsabilidade. Essa tomada de decisão automatizada não impacta apenas equipamentos, mas também vidas humanas. Portanto, garantir a confiabilidade dos dados, a transparência dos critérios utilizados e a segurança dos algoritmos é fundamental. O projeto, nesse sentido, está diretamente ligada ao compromisso de evitar riscos desnecessários, prevenir falhas críticas e proteger tanto os tripulantes quanto os recursos envolvidos.

Além disso, projetos como este contribuem para o avanço do conhecimento científico e tecnológico, promovendo inovação e inspirando novas gerações a se interessarem por áreas como engenharia, ciência de dados e computação. A busca pela exploração de Marte, por exemplo, não representa apenas um desafio técnico, mas também uma oportunidade de expandir os limites da humanidade, levantando discussões sobre colonização, adaptação humana a novos ambientes e o futuro da vida fora da Terra. Esse tipo de iniciativa fortalece o desenvolvimento social ao estimular educação, pesquisa e colaboração internacional.

No que diz respeito à sustentabilidade tecnológica, a atividade também demonstra preocupação com o uso eficiente de recursos. Ao utilizar cálculos e análises para otimizar o consumo energético e evitar desperdícios, evidencia-se uma abordagem consciente e alinhada com a preservação tanto dos recursos terrestres quanto espaciais. A sustentabilidade, nesse contexto, não se limita apenas à economia de energia, mas também envolve a criação de tecnologias duráveis, seguras e que minimizem impactos ambientais. Isso inclui desde a redução de falhas que possam gerar detritos espaciais até o uso responsável de materiais e fontes de energia.

Por fim, a atividade proposta contribui para o desenvolvimento de um pensamento crítico e analítico, essencial na formação de profissionais da área tecnológica. Mais do que interpretar dados, o projeto incentiva uma visão ampla sobre as consequências das ações humanas, tanto na Terra quanto no espaço. Assim, o trabalho reforça a importância de desenvolver soluções tecnológicas que não apenas avancem o conhecimento, mas também respeitem a vida, o meio ambiente e o futuro da humanidade.
